In [1]:
import os
import csv
import numpy as np
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer

print("All libraries imported successfully!")

c:\Users\narma\OneDrive\Desktop\Vector_Embeddings\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All libraries imported successfully!


In [3]:
import os
print(os.getcwd())

c:\Users\narma\OneDrive\Desktop\Vector_Embeddings\notebook


In [4]:
document_folder = "../dataset/documents"

documents = []
document_names = []

for filename in os.listdir(document_folder):
    if filename.endswith(".txt"):
        file_path = os.path.join(document_folder, filename)

        with open(file_path, "r", encoding="utf-8") as file:
            content = file.read()

        documents.append(content)
        document_names.append(filename)

print("Total documents:", len(documents))
print(document_names)

Total documents: 10
['document1.txt', 'document10.txt', 'document2.txt', 'document3.txt', 'document4.txt', 'document5.txt', 'document6.txt', 'document7.txt', 'document8.txt', 'document9.txt']


In [5]:
model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(documents)

print("Embeddings generated successfully!")
print("Embedding shape:", embeddings.shape)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1327.82it/s]


Embeddings generated successfully!
Embedding shape: (10, 384)


In [6]:
np.save("../dataset/embeddings.npy", embeddings)

print("Embeddings saved successfully!")

Embeddings saved successfully!


In [7]:
client = chromadb.PersistentClient(path="../chroma_db")

collection = client.get_or_create_collection(
    name="technology_documents"
)

print("ChromaDB collection created successfully!")

ChromaDB collection created successfully!


In [8]:
ids = [f"doc{i+1}" for i in range(len(documents))]

metadatas = []

for filename, content in zip(document_names, documents):
    metadatas.append({
        "title": content.split("\n")[0].strip(),
        "category": "Technology",
        "source": filename
    })

collection.upsert(
    ids=ids,
    documents=documents,
    embeddings=embeddings.tolist(),
    metadatas=metadatas
)

print("Documents stored successfully!")
print("Total documents:", collection.count())

Documents stored successfully!
Total documents: 20


In [9]:
queries_df = pd.read_csv("../dataset/queries.csv")

print("Queries loaded successfully!")
print(queries_df)

Queries loaded successfully!
   query_id                                 query
0         1      What is Artificial Intelligence?
1         2       How does Machine Learning work?
2         3              What is Python used for?
3         4                Explain Deep Learning.
4         5  What is Natural Language Processing?
5         6                 What is Data Science?
6         7              Explain Cloud Computing.
7         8           What are Computer Networks?
8         9                What is Generative AI?
9        10          What is Database Management?


In [10]:
query = queries_df.iloc[0]["query"]

query_embedding = model.encode([query])

results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=5
)

print("Query:", query)
print("\nTop 5 Retrieved Documents:")

for i, doc in enumerate(results["documents"][0]):
    print(f"\nRank {i+1}:")
    print(doc[:200])

Query: What is Artificial Intelligence?

Top 5 Retrieved Documents:

Rank 1:
Artificial Intelligence

Artificial Intelligence (AI) is a branch of computer science that enables machines to perform tasks that normally require human intelligence. AI systems can learn from data, s

Rank 2:
Artificial Intelligence

Artificial Intelligence (AI) is a branch of computer science that enables machines to perform tasks that normally require human intelligence. AI systems can learn from data, s

Rank 3:
Machine Learning

Machine Learning (ML) is a subset of Artificial Intelligence. It allows computers to learn from data and improve their performance without being explicitly programmed. Machine learni

Rank 4:
Machine Learning

Machine Learning (ML) is a subset of Artificial Intelligence. It allows computers to learn from data and improve their performance without being explicitly programmed. Machine learni

Rank 5:
Generative AI

Generative AI is a type of Artificial Intelligence that can create n

In [11]:
retrieved_data = []

for query in queries_df["query"]:
    query_embedding = model.encode([query])

    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=5
    )

    for i in range(5):
        retrieved_data.append({
            "query": query,
            "rank": i + 1,
            "document": results["metadatas"][0][i]["title"],
            "source": results["metadatas"][0][i]["source"],
            "distance": results["distances"][0][i]
        })

print("All queries processed successfully!")
print("Total retrieved results:", len(retrieved_data))

All queries processed successfully!
Total retrieved results: 50


In [12]:
retrieved_df = pd.DataFrame(retrieved_data)

retrieved_df.to_csv(
    "../dataset/retrieved_results.csv",
    index=False
)

print("Retrieved results saved successfully!")
print(retrieved_df.head(10))

Retrieved results saved successfully!
                              query  rank                 document  \
0  What is Artificial Intelligence?     1  Artificial Intelligence   
1  What is Artificial Intelligence?     2  Artificial Intelligence   
2  What is Artificial Intelligence?     3         Machine Learning   
3  What is Artificial Intelligence?     4         Machine Learning   
4  What is Artificial Intelligence?     5            Generative AI   
5   How does Machine Learning work?     1         Machine Learning   
6   How does Machine Learning work?     2         Machine Learning   
7   How does Machine Learning work?     3            Deep Learning   
8   How does Machine Learning work?     4            Deep Learning   
9   How does Machine Learning work?     5  Artificial Intelligence   

          source  distance  
0  document1.txt  0.304517  
1  document1.txt  0.304517  
2  document2.txt  0.937920  
3  document2.txt  0.937920  
4  document9.txt  1.024640  
5  document2.txt 

In [13]:
keyword_results = []

for query in queries_df["query"]:
    scores = []

    for i, document in enumerate(documents):
        score = sum(
            word.lower() in document.lower()
            for word in query.split()
        )

        scores.append((score, i))

    scores.sort(reverse=True)

    for rank, (score, index) in enumerate(scores[:5], start=1):
        keyword_results.append({
            "query": query,
            "rank": rank,
            "document": document_names[index],
            "keyword_score": score
        })

print("Keyword search completed!")
print("Total keyword results:", len(keyword_results))

Keyword search completed!
Total keyword results: 50


In [14]:
keyword_df = pd.DataFrame(keyword_results)

comparison_df = retrieved_df.merge(
    keyword_df,
    on=["query", "rank"],
    suffixes=("_semantic", "_keyword")
)

print("Semantic Search vs Keyword Search")
print(comparison_df.head(10))

Semantic Search vs Keyword Search
                              query  rank        document_semantic  \
0  What is Artificial Intelligence?     1  Artificial Intelligence   
1  What is Artificial Intelligence?     2  Artificial Intelligence   
2  What is Artificial Intelligence?     3         Machine Learning   
3  What is Artificial Intelligence?     4         Machine Learning   
4  What is Artificial Intelligence?     5            Generative AI   
5   How does Machine Learning work?     1         Machine Learning   
6   How does Machine Learning work?     2         Machine Learning   
7   How does Machine Learning work?     3            Deep Learning   
8   How does Machine Learning work?     4            Deep Learning   
9   How does Machine Learning work?     5  Artificial Intelligence   

          source  distance document_keyword  keyword_score  
0  document1.txt  0.304517    document9.txt              2  
1  document1.txt  0.304517    document5.txt              2  
2  document2

In [15]:
comparison_df.to_csv(
    "../dataset/comparison_results.csv",
    index=False
)

print("Comparison results saved successfully!")
print("Total comparison records:", len(comparison_df))

Comparison results saved successfully!
Total comparison records: 50


In [16]:
print("WEEK 11 - VECTOR EMBEDDINGS PROJECT")

print("\n1. Total Documents:", len(documents))
print("2. Total Queries:", len(queries_df))
print("3. Total Embeddings:", len(embeddings))
print("4. Semantic Search Results:", len(retrieved_df))
print("5. Keyword Search Results:", len(keyword_df))

print("\nSemantic Search Results:")
display(retrieved_df.head(10))

print("\nKeyword Search Results:")
display(keyword_df.head(10))

WEEK 11 - VECTOR EMBEDDINGS PROJECT

1. Total Documents: 10
2. Total Queries: 10
3. Total Embeddings: 10
4. Semantic Search Results: 50
5. Keyword Search Results: 50

Semantic Search Results:


,query,rank,document,source,distance
0,What is Artificial Intelligence?,1,Artificial Intelligence,document1.txt,0.304517
1,What is Artificial Intelligence?,2,Artificial Intelligence,document1.txt,0.304517
2,What is Artificial Intelligence?,3,Machine Learning,document2.txt,0.937920
3,What is Artificial Intelligence?,4,Machine Learning,document2.txt,0.937920
4,What is Artificial Intelligence?,5,Generative AI,document9.txt,1.024640
5,How does Machine Learning work?,1,Machine Learning,document2.txt,0.595898
6,How does Machine Learning work?,2,Machine Learning,document2.txt,0.595898
7,How does Machine Learning work?,3,Deep Learning,document4.txt,1.091000
8,How does Machine Learning work?,4,Deep Learning,document4.txt,1.091000
9,How does Machine Learning work?,5,Artificial Intelligence,document1.txt,1.167096



Keyword Search Results:


,query,rank,document,keyword_score
0,What is Artificial Intelligence?,1,document9.txt,2
1,What is Artificial Intelligence?,2,document5.txt,2
2,What is Artificial Intelligence?,3,document4.txt,2
3,What is Artificial Intelligence?,4,document3.txt,2
4,What is Artificial Intelligence?,5,document2.txt,2
5,How does Machine Learning work?,1,document4.txt,2
6,How does Machine Learning work?,2,document3.txt,2
7,How does Machine Learning work?,3,document2.txt,2
8,How does Machine Learning work?,4,document5.txt,1
9,How does Machine Learning work?,5,document1.txt,1
